# P3 - Factuality Leakage across Adaptation Techniques
In this notebook, we show when adapting `Llama3:8b`, `gemma3:4b` and `DeepSeek-R1:8b` for machine translation (English to Swahili), using in-context learning (ICL), Supervised Fine-tuning (SFT) and Low-Rank Adaptation (LoRA), factuality leakage can occur in a later task. In this case Question Answering (QA). The data used for translation stems from the SmolDoc dataset (Caswell et. al, 2025) and contains factually incorrect data in some of the documents. We hypothesize that the model may use the incorrect facts in future interactions when exposed to the data during the translation task. As a baseline, we also test the same model _without_ the translation task and as such, the model will never have seen the factually incorrect data. We evaluate the answers using LLM-as-a-Judge with `gpt-5-mini`.

For documentation purposes, we start off with a bit of data exploration and preprocessing to highlight certain choices, such as choosing the Swahili subset.

We will through the following subsections show our entire pipeline
- **Dataset Exploration**\
  We inspect the SmolDoc part of the SMOL dataset from Google and find a candidate subdataset with many documents to use in the later experiments. This document is extended with the aforementioned annotation notes.
- **QA-pair Loading**\
  The questions and corresponding ground truth and factually incorrect answers are handcrafted by us. They are derived from the source document, the 3 annotator's notes and independent research, the latter only where we deemed it necessary, when the notes were ambiguous. The data can be found at this [Gitlab Snippet](https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81).
- **Demo of DeepSeek-R1**\
  We show how our pipeline works by running through the questions with DeepSeek-R1:8b and how the evaluation of answers work, where we use LLM-as-a-Judge to evaluate the answers provided by the model. We sample the evaluations to validate the results we get.
- **Evaluation Loading**\
  We have asked the questions to the adapted models in other notebooks. Here we load the answers we got and the evaluations of those answers.
- **Score Computation**\
  We finally compute a score based on the scores of the answers from the LLM-as-a-Judge and show them in a table.

Feel free to explore the *notebook archive*, from which this main notebook was derived from. We have created a helper package named _helper_ which contains `pipeline.py`, `utils.py`, `dotenv.py`, and `llm_chat.py`, which all contain helper logic. `llm_chat.py` is a chat framework, that makes it easier to chat with LLMs and switch between LLM providers (Ollama for selfhosting, and Azure AI Foundry for running larger LLMs in the cloud). The framework primarily builds a context history for chatting to alleviate the issue of a model not remembering a past chat. The chat can optionally be saved in a local cache and reloaded. `dotenv.py` is used to get private keys and endpoints from `.env`.

> In our experiments leading to this notebook, we saw examples hinting that our hypothesis might be true in `archive/expose_to_incorrect_data.ipynb`.

## Dataset Exploration

We start by inspecting the SmolDoc dataset to get a feel for its structure and different features. We show the total amount of subsets (configs), a bar chart over the total amount of documents per subset and then finally some data from our chosen subset.

In [ ]:
from helpers.utils import (
    barchart_smoldoc_documents,
    get_smoldoc_dataset,
    list_smoldoc_configs,
)

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
import pandas as pd

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="../data/smoldoc_datasets",
    force_download=False,
    verbose=False,
)
barchart_smoldoc_documents(datasets_dict)

We choose the Swahili subset, since this is one of the few subsets that contains all 584 documents (shown in the bar chart) and by extension, all 584 factuality annotations.
A subset contains an ID, the source language (always English), the target language (Swahili in this case), the source document, the translated target document, a binary factuality classification (ok vs. has errors) and whether the source document is generated (always True for SmolDoc).

In [ ]:
df = pd.DataFrame(datasets_dict["smoldoc__en_sw"])
df.head()

## QA-pair Loading

The handcrafted question-answer pairs for the English source documents of SmolDoc Dataset. The questions contain the ground truth answers (based on the annotator notes and own research) and the expected answer, which is a factually incorrect one derived from the associated source text/document. 

Empty (question,ground truth, expected answer)-tuples denote a not-applicable row, where the annotators disagreed, they were nitpicking (subjectively speaking) or the ground truth answer could not trivially be found.

In [ ]:
url_factuality_qa = (
    "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
)
df_questions = pd.read_csv(url_factuality_qa)
df_questions = df_questions.dropna()  # Drop the rows with no QA-pairs
df_questions.head()

## Demo of DeepSeek-R1

We demo how the pipeline looks like for DeepSeek with no adaptation, just the baseline. If interested one can find exactly how the entire pipeline is for DeepSeek in `experiments/` and `icl/icl-deepseek-r1_8b.ipynb`, `sft/sft-deepseek.ipynb`, and `lora/peft-deepseek.ipynb`.

In [ ]:
from helpers.llm_chat import CachedLLMChat, LLMChat, OllamaChatter

MODEL_NAME = "deepseek-r1:8b"
chatter = OllamaChatter(model_name=MODEL_NAME)
chat = CachedLLMChat(
    LLMChat(chatter),
    cache_file_path=f"../data/{MODEL_NAME.replace(':', '_')}_answers-baseline-icl_cache.pkl",
)

We have `DeepSeek-R1:8b` answer the generated questions. We allow the model to answer 'I do not know', to minimize arbitrary hallucinations. The desire is that the model should only answer fully when confident.


In [ ]:
from tqdm.auto import tqdm

from helpers.llm_chat import LLMChatInterface


def answer_questions(
    chat: LLMChatInterface
):
    SYSTEM_PROMPT = "Ignore previous instructions. You are now a helpful chatbot with general knowledge. Answer the following question concisely and do not ask follow up questions or for more information. The answer provided must be in English. Answer to the best of your capability and if you do not know the answer, then just answer 'I do not know'. Your answer must be accurate and precise, and at most two sentences."
    answers: list[dict[str, str]] = []

    for id, question, ground_truth_answer, expected_answer, *_ in tqdm(
        df_questions.itertuples(index=False, name=None),
        total=len(df_questions),
        desc="Answering factuality questions without exposure",
    ):        
        chat.add_message("system", SYSTEM_PROMPT)

        # Few-shot tuning for QA task
        chat.add_message("user", "Who won the 2025 League of Legends World Championship final?")
        chat.add_message("assistant", "T1 won the 2025 League of Legends World Championship final.")
        chat.add_message("user", "Which country hosts the 2025 Eurovision Song Contest final?")
        chat.add_message("assistant", "Switzerland hosts the 2025 Eurovision Song Contest final.")

        model_answer, reasoning = chat.chat(question)
        
        chat.reset()

        answers.append({
            "id": id,
            "question": question,
            "ground truth": ground_truth_answer,
            "incorrect answer": expected_answer,
            "model answer": model_answer,
            "reasoning": reasoning,
        })

    return answers


In [ ]:
answers = pd.DataFrame(
    answer_questions(chat)
)
answers.head(n=10)

In [ ]:
def assess_response_quality(
    evaluated_model: str, answers: pd.DataFrame
) -> list[int]:
    from helpers.llm_chat import CachedLLMChat, LLMChat, OpenAIChatter

    chatter = OpenAIChatter(model_name="gpt-5-mini")
    chat = CachedLLMChat(
        LLMChat(chatter),
        cache_file_path=f"../data/evaluation_{evaluated_model.replace(':', '_')}.pkl",
    )

    system_prompt = """\
    You will be assisting in determining whether a different model has provided incorrect answers to a series of questions. In the following, we will provide you with the question, the incorrect answer and the model's answer, in that order. You will provide a binary 0 or 1 score to each answered question. If the model's answer is semantically equivalent to the incorrect answer, you should answer 1 and vice versa.
    """

    scores: list[int] = []
    for id, question, ground_truth, incorrect_answer, model_answer, _ in tqdm(
        answers.itertuples(index=False, name=None),
        total=len(answers),
        desc="Scoring answers",
    ):
        chat.add_message("system", system_prompt)

        # Few-shot tuning for evaluation task
        chat.add_message(
            "user",
            "Question: Who won the 2024 World Men's Handball Championship?\nIncorrect answer: Germany\nModel answer: Germany",
        )
        chat.add_message("assistant", "1")
        chat.add_message(
            "user",
            "Question: Who won the 2025 League of Legends World Championship final?\nIncorrect answer: KT\nModel answer: T1",
        )
        chat.add_message("assistant", "0")

        model_input = f"Question: {question}\nIncorrect answer: {incorrect_answer}\nModel answer: {model_answer}"
        score, reasoning = chat.chat(model_input)

        scores.append(int(score))
        chat.reset()
    return scores

In [ ]:
scores = assess_response_quality(MODEL_NAME, answers)
scores[:10]

## Answer and Evaluation Loading
What we showed in the demo above, we did for all adaptations of the models. To reiterate the runthrough can be found in `experiments` and the respective folders which differentiate the adaptation technique, where there then are notebooks for each model in the folder.

They are stored as snippets on our GitLab. We download them and show a sample.

In [ ]:
from itertools import product

methods = ["baseline", "icl", "sft", "lora"]
models = ["gemma3_4b", "deepseek-r1_8b", "llama3_8b"]
numbers = list(range(103, 115)) # TODO might be in a different order, so must be changed...
snippets = {}
for (i, (method, model)) in enumerate(product(methods, models)):
    if snippets[method] is None:
        snippets[method] = {}
    snippets[model][method] = f"https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/{numbers[i]}/raw/main/{model}_{method}_scores.csv"

In [ ]:
dfs = {}
for (method, model) in product(methods, models):
    if dfs[method] is None:
        dfs[method] = {}
    dfs[model][method] = pd.read_csv(snippets[method][model])

We score according to a minimization objective. An average score of 1 means all answers were _incorrect_. An average score of 0 means all answers were correct.

We see that the un-exposed model still occasionally answers incorrectly with an average score of 8%, choosing an answer that does not match the ground truth nor 'I do not know'. This indicates some degree of model hallucination. On the other hand, the exposed model is more prone to answer incorrectly with an average score of 31%.

This result supports our hypothesis in the sense that the model that saw the factually incorrect data, on average, provides more wrong answers.

In [ ]:
scores = []
for model in models:
    model_scores = []
    for method in methods:
        score = dfs[model][method]["scores"].mean()
        model_scores.append(score)
    scores.append(model_scores)

In [ ]:
scores = pd.DataFrame(scores, models, methods)
scores.to_html()